# TPCx-AI UC8: Trip Type Classification with Distributed XGBoost

Multi-class classification on retail order data using the TPCx-AI benchmark (unofficial).

**Pipeline:** Snowflake Tables → Snowpark SQL (join + aggregate) → Distributed XGBoost

## Prerequisites

- A **Snowflake Notebook on Container Runtime (CPU)** — `scale_cluster`, `XGBEstimator`, and `DataConnector` are only available there.
- A compute pool sized for the largest scale factor you plan to run (`CPU_X64_M`, up to 12 nodes for scale factor 1000).
- TPCx-AI UC8 data loaded into `TPCXAI_V2.{SF1, SF10, SF100, SF1000}.{ORDERS, LINEITEM, PRODUCT}` — one schema per scale factor, matching the TPCx-AI kit's output layout. See [`setup_snowflake.sql`](./setup_snowflake.sql) for the load script; generate the raw CSVs with the official [TPCx-AI kit](https://www.tpc.org/tpcx-ai/).

## Running

1. Open this notebook in a Container Runtime notebook attached to your compute pool.
2. In the config cell, adjust `SCALE_FACTORS`, `NODES_MAP`, and `RESULT_TABLE`.
3. Run all cells. Per-run timings are printed inline and appended to `RESULT_TABLE`.

## Pipeline Components

| Component | What it does |
|---|---|
| `session.table()` | Reads from Snowflake tables (loaded via `setup_snowflake.sql`) |
| Snowpark SQL | Three-way join + GROUP BY for feature engineering (weekday one-hot + per-department quantity sums) |
| `DENSE_RANK()` | Encodes trip-type labels to 0-based indices |
| `DataConnector` | Bridges Snowpark DataFrame to distributed XGBoost |
| `XGBEstimator` | Distributed XGBoost training across Ray workers |

In [ ]:
import timeit, re
from datetime import datetime

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window
from snowflake.ml.data.data_connector import DataConnector
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
from snowflake.runtime._version import __version__ as MLRUNTIME_VERSION

session = get_active_session()

In [ ]:
# ---- Config ----
DATABASE      = "TPCXAI_V2"                      # set up in setup_snowflake.sql
SCALE_FACTORS = [1, 10, 100, 1000]               # which scale factors to run; SF1000 needs scale factor 1000 data + 12 nodes
NODES_MAP     = {1: 1, 10: 2, 100: 4, 1000: 12}  # cluster size per scale factor
NUM_RUNS      = 5                                # repeat each scale factor for variance reporting

# Where benchmark timings are appended (one row per phase per run).
# TODO: change to a database/schema you can write to.
RESULT_TABLE  = "<YOUR_DB>.<YOUR_SCHEMA>.UC8_RESULTS"

WEEKDAYS = ["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY", "SUNDAY"]

# 68 product departments from the TPCx-AI UC8 spec; one summed-quantity feature per department.
DEPARTMENTS = [
    "FINANCIAL SERVICES", "SHOES", "PERSONAL CARE", "PAINT AND ACCESSORIES", "DSD GROCERY",
    "MEAT - FRESH & FROZEN", "DAIRY", "PETS AND SUPPLIES", "HOUSEHOLD CHEMICALS/SUPP",
    "IMPULSE MERCHANDISE", "PRODUCE", "CANDY, TOBACCO, COOKIES", "GROCERY DRY GOODS",
    "BOYS WEAR", "FABRICS AND CRAFTS", "JEWELRY AND SUNGLASSES", "MENS WEAR", "ACCESSORIES",
    "HOME MANAGEMENT", "FROZEN FOODS", "SERVICE DELI", "INFANT CONSUMABLE HARDLINES",
    "PRE PACKED DELI", "COOK AND DINE", "PHARMACY OTC", "LADIESWEAR", "COMM BREAD", "BAKERY",
    "HOUSEHOLD PAPER GOODS", "CELEBRATION", "HARDWARE", "BEAUTY", "AUTOMOTIVE",
    "BOOKS AND MAGAZINES", "SEAFOOD", "OFFICE SUPPLIES", "LAWN AND GARDEN", "SHEER HOSIERY",
    "WIRELESS", "BEDDING", "BATH AND SHOWER", "HORTICULTURE AND ACCESS", "HOME DECOR", "TOYS",
    "INFANT APPAREL", "LADIES SOCKS", "PLUS AND MATERNITY", "ELECTRONICS",
    "GIRLS WEAR, 4-6X  AND 7-14", "BRAS & SHAPEWEAR", "LIQUOR,WINE,BEER",
    "SLEEPWEAR/FOUNDATIONS", "CAMERAS AND SUPPLIES", "SPORTING GOODS",
    "PLAYERS AND ELECTRONICS", "PHARMACY RX", "MENSWEAR", "OPTICAL - FRAMES",
    "SWIMWEAR/OUTERWEAR", "OTHER DEPARTMENTS", "MEDIA AND GAMING", "FURNITURE",
    "OPTICAL - LENSES", "SEASONAL", "LARGE HOUSEHOLD GOODS", "ONE HR PHOTO",
    "CONCEPT STORES", "HEALTH AND BEAUTY AIDS",
]

safe = lambda s: re.sub(r'[^0-9A-Za-z]+', '_', s).upper().strip('_')
DEPT_COLS = [safe(d) for d in DEPARTMENTS]

## Run Benchmark

Iterates over all scale factors. For each SF: scales the cluster, runs preprocessing + training `NUM_RUNS` times, and writes results to the result table.

In [ ]:
from snowflake.ml.runtime_cluster import scale_cluster
from snowflake.snowpark import Row

phases = ["preprocess", "train"]
all_rows = []

for SF in SCALE_FACTORS:
    SCHEMA = f"SF{SF}"
    NUM_NODES = NODES_MAP[SF]
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"\n{'='*60}")
    print(f"SF={SF}  |  {NUM_NODES} node(s)  |  {NUM_RUNS} run(s)")
    print(f"{'='*60}")

    # Scale cluster
    scale_cluster(NUM_NODES)
    print(f"Cluster scaled to {NUM_NODES} nodes")

    for run in range(1, NUM_RUNS + 1):
        timings = {}
        print(f"\n--- SF={SF} Run {run}/{NUM_RUNS} ---")

        # ---- Preprocess ----
        start = timeit.default_timer()

        orders   = session.table(f"{DATABASE}.{SCHEMA}.ORDERS")
        lineitem = session.table(f"{DATABASE}.{SCHEMA}.LINEITEM")
        product  = session.table(f"{DATABASE}.{SCHEMA}.PRODUCT")

        raw = (
            lineitem
            .join(orders, lineitem["LI_ORDER_ID"] == orders["O_ORDER_ID"])
            .join(product, lineitem["LI_PRODUCT_ID"] == product["P_PRODUCT_ID"])
            .filter(F.col("O_ORDER_ID").is_not_null())
            .filter(F.col("TRIP_TYPE") != 14)
        )

        # weekday one-hot + per-department quantity sums
        agg_exprs = [
            F.sum("QUANTITY").alias("SCAN_COUNT"),
            F.sum(F.abs(F.col("QUANTITY"))).alias("SCAN_COUNT_ABS"),
            F.min("TRIP_TYPE").alias("TRIP_TYPE"),
        ]
        for day in WEEKDAYS:
            agg_exprs.append(
                F.max(F.when(F.upper(F.col("WEEKDAY")) == day, 1).otherwise(0)).alias(day)
            )
        for dept, col_name in zip(DEPARTMENTS, DEPT_COLS):
            agg_exprs.append(
                F.sum(F.when(F.col("DEPARTMENT") == dept, F.col("QUANTITY")).otherwise(0)).alias(col_name)
            )

        features_df = raw.group_by("O_ORDER_ID").agg(*agg_exprs).drop("O_ORDER_ID").fillna(0)
        features_df = features_df.with_column(
            "TRIP_TYPE",
            F.dense_rank().over(Window.order_by("TRIP_TYPE")) - 1
        )
        features_df = features_df.cache_result()
        num_classes = features_df.select(F.max("TRIP_TYPE")).collect()[0][0] + 1
        row_count = features_df.count()
        input_cols = [c for c in features_df.columns if c != "TRIP_TYPE"]

        timings["preprocess"] = timeit.default_timer() - start
        print(f"  Preprocess: {timings['preprocess']:.1f}s  |  {row_count:,} orders  |  {num_classes} classes")

        # ---- Train ----
        start = timeit.default_timer()

        connector = DataConnector.from_dataframe(features_df)
        estimator = XGBEstimator(
            params={"tree_method": "hist", "num_class": num_classes},
            scaling_config=XGBScalingConfig(use_gpu=False),
            objective="multi:softmax",
            n_estimators=100,
        )
        booster = estimator.fit(connector, input_cols=input_cols, label_col="TRIP_TYPE")

        timings["train"] = timeit.default_timer() - start
        print(f"  Train: {timings['train']:.1f}s  |  {len(input_cols)} features")

        # Record results
        total = sum(timings[p] for p in phases)
        print(f"  Total: {total:.1f}s")

        for phase in phases:
            all_rows.append(Row(
                TIMESTAMP=ts, SF=SF, NUM_NODES=NUM_NODES,
                RUN=run, PHASE=phase, ELAPSED_S=round(timings[phase], 2),
                MLRUNTIME_VERSION=MLRUNTIME_VERSION,
            ))

    # Write after each scale factor completes (in case later ones fail)
    session.create_dataframe(all_rows).write.mode("append").save_as_table(RESULT_TABLE)
    print(f"\nResults for SF={SF} written to {RESULT_TABLE}")
    all_rows = []

print(f"\n{'='*60}")
print("All benchmarks complete!")
print(f"{'='*60}")